# 03_train_hog_svm.ipynb

Baseline truyền thống HOG + SVM trên DurianLDD. Chạy trên CPU, cùng split khóa với các baseline học sâu, báo macro F1 trên validation. Không tối ưu siêu tham số.

Cell môi trường kéo nhánh `GIT_BRANCH` và gắn ảnh từ `cthng123/durian-leaf-disease-dataset`. Split lấy từ `manifests/split_manifest.csv`. Tập test không mở.

## 1. Môi trường

Internet để On. Accelerator None. Dataset input là `cthng123/durian-leaf-disease-dataset`.

In [ ]:
## 2. Dữ liệu và cấu hình

Ảnh đọc từ dataset Kaggle. Tập train và validation đọc từ cột `split` của manifest.

## 2. Dữ liệu và cấu hình

Ảnh xám 224×224, không augmentation. Tập train và validation đọc từ cột `split` của manifest.

In [ ]:
from src.training.run import load_configs

CONFIG = "configs/train_hog_svm.yaml"
train_cfg, pipeline_cfg = load_configs(CONFIG)

print(f"Seed: {train_cfg['seed']}")
print(f"HOG: {train_cfg['hog']}")
print(f"SVM: {train_cfg['svm']}")
print("Thiết bị: cpu")
print("Tập test: không mở")

## 3. Trích đặc trưng, train và kết quả validation

Trích HOG cho train và validation, chuẩn hóa, fit SVM, rồi tính macro F1, balanced accuracy, recall từng lớp và confusion matrix. Mất khoảng bốn phút trên CPU.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.training.hog_svm import main

paths = main(CONFIG)
record = pd.read_json(paths["metrics"], typ="series")
display(pd.DataFrame([{
    "val_macro_f1": record["val_macro_f1"],
    "val_balanced_accuracy": record["val_balanced_accuracy"],
    "val_accuracy": record["val_accuracy"],
    "feature_dimension": record["feature_dimension"],
    "seconds": record["elapsed_seconds"],
}]))
display(pd.DataFrame(
    [{"class_name": name, "recall": value} for name, value in record["per_class_recall"].items()]
))
display(Image(filename=paths["confusion_matrix"]))